# Análise de Acessibilidade

**AUTORIA:** [REDE MOB](https://www.redemob.com.br/)

Análise do padrão de viagens de um município com base em dados de pesquisa de origem e destino. As linhas de código abaixo têm como objeto o município de Belo Horizonte, mas serve para qualquer outro, desde que os dados de entrada estejam no layout padrão da Plataforma.

**PANORAMA:**
- #TODO: Listar as análises aqui contidas

**MAIS INFORMAÇÕES:**
- [Layout da Plataforma]
- [Sumário dos Dados Disponíveis]
- *Lorem ipsum: Conteúdo do MOB de interesse, técnico ou de divulgação*

**LINKS DE INTERESSE:**
- links para materiais técnicos e acadêmicos gerais de referência a respeito do conteúdo abordado



# Introdução

Este script foi concebido em caráter de tutorial, tomando como exemplo o município de Belo Horizonte/MG, de forma que as considerações e discussões aqui contidas foram tecidas no contexto dessa municipalidade. Com efeito, procurou-se, na medida do possível, deixar o texto abrangente a ponto de orientar as análises de outros municípios. Ou seja, este script acaba constituindo um apoio para o diagnóstico de outras localidade, na medida em que houve um esforço de trazer técnicas e elementos norteadores para contribuir para o diagnóstico de outros locais. Com efeito, ao alterar os parâmetros de entrada, conforme demonstrado logo abaixo, podem ser gerados mapas e gráficos de territórios distintos. Nesse caso, as considerações originalmente tecidas para Belo Horizonte podem não se aplicar completamente, mas, elas ainda devem fornecer insumos para a interpretação de resultados de outros locais.

# Instruções

Este script precisa de dois grupos de dados:
1. Dados de origem e destino #TODO: explicar como obter
2. Malhas territoriais de zonas de tráfego

Essses dados devem ser atribuídos às variáveis abaixo. Em seguida, deve-se rodar todo o script e os resultados estarão ao final.

# Backend

In [1]:
import datetime as dt
from itertools import islice
from joblib import Parallel, delayed
import os
import pathlib
from time import perf_counter
from typing import List, Tuple, Iterable

import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandana as pdna
import pandas as pd
import partridge as ptg
import seaborn as sns
from tqdm import tqdm


%matplotlib inline
%config InlineBackend.figure_format='retina'

pd.options.display.float_format = '{:,.2f}'.format

ModuleNotFoundError: No module named 'IPython.core.pylabtools'

In [ ]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder)

## INPUTS

In [ ]:
inpath = out_folder / 'A/hexagonos.gpkg'

hexes = gpd.read_file(inpath, layer='socioeconomico')

In [ ]:
hexes.head(3)

## Network

In [ ]:
def get_study_area(ibge_id):
    if not isinstance(ibge_id, list):
        ibge_id = [ibge_id]

    return pd.concat([
        geobr.read_municipality(id_)
        for id_
        in ibge_id
        ]).reindex(columns=['geometry'])


place = get_study_area(ibge_id=3106200) # Belo Horizonte

In [ ]:
crs = place.estimate_utm_crs(datum_name='SIRGAS 2000')
crs

In [ ]:
def get_network(
        place,
        network_type='walk',
        out_crs=31983,
        is_simplified=True,
        consolidate_intersections=False,
        intersection_consolidation_tolerance=15,
        ):
    graph = ox.graph_from_polygon(
        place.to_crs(4326).union_all().convex_hull,
        network_type=network_type,
        simplify=is_simplified,
        )

    if consolidate_intersections:
        graph = ox.consolidate_intersections(
            graph,
            rebuild_graph=True,
            tolerance=intersection_consolidation_tolerance,
            dead_ends=True,
            )

    graph = consolidate_edges(graph)

    return ox.project_graph(graph, to_crs=out_crs)


def consolidate_edges(G):
    invalid_bunch = _select_invalid_edges(G)
    if invalid_bunch:
        G.remove_edges_from(invalid_bunch)
        G.remove_nodes_from(
            list(nx.isolates(G))
            )

    return get_largest_component(G)


def get_largest_component(G):
    largest_wcc = max(nx.weakly_connected_components(G), key=len)
    
    largest_graph = G.__class__()
    largest_graph.add_nodes_from((n, G.nodes[n]) for n in largest_wcc)
    if largest_graph.is_multigraph():
        largest_graph.add_edges_from(
            (n, nbr, key, d)
            for n, nbrs in G.adj.items()
            if n in largest_wcc
            for nbr, keydict in nbrs.items()
            if nbr in largest_wcc
            for key, d in keydict.items()
        )
    else:
        largest_graph.add_edges_from(
            (n, nbr, d)
            for n, nbrs in G.adj.items()
            if n in largest_wcc
            for nbr, d in nbrs.items()
            if nbr in largest_wcc
        )
    largest_graph.graph.update(G.graph)

    return largest_graph


def _select_invalid_edges(G):
    """
    Removes edges of certain highway types that are
    not really part of day to day paths, such as trails.
    """
    h_types = [
        'tracks',
        'unclassified',
        'steps',
        'path',
        'corridor',
        'track',
        ]
    return [
        tuple(e)
        for *e, d
        in G.edges(data=True, keys=True)
        if np.isin(h_types, d.get('highway')).any() 
        ]


In [ ]:
graph = get_network(
    place,
    network_type='walk',
    out_crs=crs,
    is_simplified=False,
    consolidate_intersections=False,
    #intersection_consolidation_tolerance=15,
    )

A seguir são imputados na rede as velocidades de caminhada médias, derivadas do [passo usual de adultos aparentemente saudáveis](https://d-nb.info/1223247600/34). Esse estudo demonstra que a velocidade usual é de 1,31 (1,27–1,35) m/s. Nesse sentido, assim como [Boeing et al. (2022)](https://www.thelancet.com/journals/langlo/article/PIIS2214-109X(22)00072-9/fulltext), **resolvemos adotar o limite inferior de 1,27 m/s**.

In [ ]:
# Imputa tempos de viagem na rede
for u, v, data in graph.edges(data=True):
    data['speed_kph'] = 1.27 * 3.6 # m/s → km/h
graph = ox.add_edge_travel_times(graph)

In [ ]:
# Constrói rede do Pandana
nodes = ox.graph_to_gdfs(graph, edges=False)[['x', 'y']]
edges = ox.graph_to_gdfs(graph, nodes=False).reset_index()[['u', 'v', 'travel_time']]

network = pdna.Network(
    node_x=nodes['x'],
    node_y=nodes['y'], 
    edge_from=edges['u'],
    edge_to=edges['v'],
    edge_weights=edges[['travel_time']]
)

In [ ]:
def compute_od_chunk(
        origin_chunk: np.ndarray,
        dest_chunk: np.ndarray,
        max_cost: int,
        imp_name: str,
        network: pdna.Network,
        task_id: int,
        ) -> pd.DataFrame:
    start = perf_counter()
    
    from_ids, to_ids = np.meshgrid(origin_chunk, dest_chunk, indexing="ij")
    from_ids = from_ids.ravel()
    to_ids = to_ids.ravel()

    costs = network.shortest_path_lengths(from_ids, to_ids, imp_name=imp_name)
    
    df = pd.DataFrame({
        "from_id": from_ids,
        "to_id": to_ids,
        imp_name: np.array(costs) / 60  # convert seconds to minutes
        })

    result = (
        df
        .loc[(df["from_id"] != df["to_id"]) & (df[imp_name] <= max_cost)]
        .set_index(["from_id", "to_id"])
        )

    duration = perf_counter() - start
    print(f"🕒 Batch in Task {task_id} processed in {duration/60:.2f} min")

    return result


def od_task_generator(
        origins: np.ndarray,
        destinations: np.ndarray,
        origin_batch_size: int,
        dest_batch_size: int
    ) -> Iterable[Tuple[np.ndarray, np.ndarray]]:
    for i in range(0, len(origins), origin_batch_size):
        o_batch = origins[i:i + origin_batch_size]
        for j in range(0, len(destinations), dest_batch_size):
            d_batch = destinations[j:j + dest_batch_size]
            yield o_batch, d_batch


def compute_od_matrix_streaming(
        origins: gpd.GeoDataFrame,
        destinations: gpd.GeoDataFrame,
        network: pdna.Network,
        imp_name: str = "travel_time",
        max_cost: int = 120,
        origin_batch_size: int = 1000,
        dest_batch_size: int = 1000,
        n_jobs: int = -1,
        verbose: bool = True
        ) -> pd.DataFrame:
    # Get closest network nodes
    origin_ids = network.get_node_ids(
        origins.centroid.x, origins.centroid.y
        )
    dest_ids = network.get_node_ids(
        destinations.centroid.x, destinations.centroid.y
        )

    # Pre-calculate total number of tasks for progress bar
    n_origin_chunks = (len(origin_ids) + origin_batch_size - 1) // origin_batch_size
    n_dest_chunks = (len(dest_ids) + dest_batch_size - 1) // dest_batch_size
    total_tasks = n_origin_chunks * n_dest_chunks

    task_gen = od_task_generator(origin_ids, dest_ids, origin_batch_size, dest_batch_size)

    results = Parallel(n_jobs=n_jobs, backend="threading")(
        delayed(compute_od_chunk)(o, d, max_cost, imp_name, network, task_id)
        for task_id, (o, d) in enumerate(
            tqdm(task_gen, total=total_tasks, desc="Computing OD matrix")
            )
        )

    return pd.concat(results)


In [ ]:
origins = hexes.loc[hexes.aperture == 9].copy()
destinations = hexes.loc[hexes.aperture == 9].copy()

In [ ]:
costs = compute_od_matrix_streaming(
    origins,
    destinations,
    network,
    imp_name="travel_time",
    max_cost=90,
    origin_batch_size=1500,
    dest_batch_size=len(destinations),
    n_jobs=3,
    )

In [ ]:
out_folder

In [ ]:
def batched(
        iterable: Iterable[T],
        batch_size: int
        ) -> Iterator[List[T]]:
    it = iter(iterable)
    while batch := list(islice(it, batch_size)):
        yield batch

def compute_batch(
        origin_batch: List[int],
        dest_ids: List[int],
        max_cost: int,
        imp_name: str,
        ) -> pd.DataFrame:
    """
    Compute shortest path lengths for a batch of origins using global 'network'.
    """
    n_orig = len(origin_batch)
    n_dest = len(dest_ids)

    from_ids = np.repeat(origin_batch, n_dest)
    to_ids = np.tile(dest_ids, n_orig)
    
    distances = network.shortest_path_lengths(
        from_ids, to_ids, imp_name=imp_name
        )
    batch_matrix = pd.DataFrame({
            "from_id": from_ids,
            "to_id": to_ids,
            imp_name: np.array(distances) / 60 # secs to minutes
        })
    return (
        df
        .loc[
            (df["from_id"] != df["to_id"])
            & (df[imp_name] <= max_cost)
            ]
        .set_index(['from_id', 'to_id'])
    )


def compute_od_matrix_batched(
        origins: gpd.GeoDataFrame,
        destinations: gpd.GeoDataFrame,
        network: pdna.Network,  # assumed to be global, used for OSM ID mapping
        imp_name: str = "travel_time",
        max_cost: int = 120,
        reindex_name: Optional[str] = None,
        batch_size: int = 500,
        n_jobs: int = -1
        ) -> pd.DataFrame:
    """
    Compute an OD cost matrix in parallel using batched shortest path queries.

    Parameters
    ----------
    origins : gpd.GeoDataFrame
        Points of origin with geometries (must be in network's CRS).
    destinations : gpd.GeoDataFrame
        Points of destination with geometries (must be in network's CRS).
    network : pandana.Network
        Pandana routing network (used only to get OSM IDs here).
    imp_name : str
        Impedance column name used in routing (e.g., "travel_time").
    reindex_name : Optional[str]
        Optional column in `origins` to use as the DataFrame index and columns.
    batch_size : int
        Number of origins per batch.
    n_jobs : int
        Number of parallel jobs (e.g., -1 = all cores).

    Returns
    -------
    pd.DataFrame
        OD matrix where index = origins, columns = destinations.
    """
    origins = origins.copy()
    destinations = destinations.copy()

    # Get closest network nodes
    origins["osm_ids"] = network.get_node_ids(
        origins.centroid.x, origins.centroid.y
        )
    destinations["osm_ids"] = network.get_node_ids(
        destinations.centroid.x, destinations.centroid.y
        )

    origin_ids: List[int] = origins["osm_ids"].tolist()
    dest_ids: List[int] = destinations["osm_ids"].tolist()

    # Create batches of origin IDs
    origin_batches = batched(origin_ids, batch_size)

    # Compute shortest paths in parallel using threading backend
    results = Parallel(n_jobs=n_jobs, backend="threading")(
        delayed(compute_batch)(batch, dest_ids, max_cost, imp_name)
        for batch in tqdm(origin_batches, desc="Computing OD matrix", total=len(origin_ids) // batch_size + 1)
        )

    # Combine all batch results into one OD matrix
    od_matrix = pd.concat(results, axis='columns')

    # Set index and columns if requested
    if reindex_name:
        od_matrix.index = origins[reindex_name].values
    else:
        od_matrix.index = origins.index

    if len(od_matrix.index) == len(od_matrix.columns):
        od_matrix.columns = od_matrix.index

    return od_matrix

In [13]:
origins = hexes.loc[hexes.aperture == 11].copy()
origins.shape

(1431000, 12)

In [ ]:
travel_cost_matrix = compute_od_matrix_batched(
    origins=origins,
    destinations=origins,
    network=network,
    max_cost=120,
    imp_name="travel_time",
    batch_size=500,
    )

In [ ]:
def compute_travel_cost_matrix(origins, destinations, network, imp_name=None, reindex_name=None):
    origins["osm_ids"] = network.get_node_ids(origins.centroid.x, origins.centroid.y)
    
    destinations["osm_ids"] = network.get_node_ids(
        destinations.centroid.x, destinations.centroid.y
    )
    
    ods = {}
    
    with tqdm(total=len(origins["osm_ids"])) as pbar:
        for origin in origins["osm_ids"]:
            ods[f"{origin}"] = network.shortest_path_lengths(
                [origin] * len(origins), destinations["osm_ids"],
                imp_name=imp_name,
            )
            pbar.update(1)
    
    if reindex_name:
        df = pd.DataFrame(ods, index=origins[reindex_name])
        df.columns = df.index
    else:
        df = pd.DataFrame(ods, index=origins)
    
    return df

In [ ]:
travel_cost_matrix = compute_travel_cost_matrix(origins, origins, network, imp_name='travel_time')

# Read Data

## Hexagons with  Land Uses

In [ ]:
def _get_path_to_gpkg(res):
    # TO DO: allow for more flexibility in the naming of the file?
    re_path = r'(BH_)(hex_\d{1,2})(_with_land_uses\.gpkg)'

    for file in (out_folder / 'A').iterdir():
        match = re.search(re_path, file.name)
        # If there's no search result, match is None
        if match:
            # The gdf in each layer was indexed by a H3 label,
            # and the index name corresponded to the hex resolution.
            # When saving to geopackage, the index name is lost.
            # This parameter takes the info on resolution from the
            # file to later insert it back to the data
            hex_res = match.group(2) # .group() is one-indexed
            
            if hex_res == f'hex_{res}':
                return file, hex_res


def _get_hexagons_with_uses(path, layername, hex_resolution):
    # The following is just because Fiona is emmiting an annoying
    # warning message that is most likely useless (see link for
    # this issue below)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore',
                                category=RuntimeWarning,)
        hex_ = gpd.read_file(path, layer=layername)
        
    hex_['year'] = layername
    
    
    return hex_.rename(columns={'index': hex_resolution})


def _drop_nulls(gdf, hex_res):
    nulls = gdf.loc[gdf.use.isnull(), hex_res].unique()
    
    
    return gdf.loc[~gdf[hex_res].isin(nulls)]
    


def _enforce_preferred_order(data):
    categoricals = {
        'category': ['active', 'passive', 'static'],
        
        'use': ['residential', 'subnormal', 'retail/services',
                'mixed', 'vacant', 'amenities', 'public services',
                'industry', 'infrastructure']
                   }
    
    for each in ['category', 'use']:
        data[each] = pd.Categorical(data[each],
                                    categories=categoricals[each],
                                    ordered=True,)
        
        
    return data.sort_values(['category', 'use'])


def get_land_use_maps(res):
    """Reads geopackage containing land use maps for 
    each study year. It assumes one year per layer and
    that each year has the same structure.
    """
    path, hex_res =  _get_path_to_gpkg(res)
    
    hex_series = []
    for layername in fiona.listlayers(path):
        hex_series.append(_get_hexagons_with_uses(path, layername, hex_res))
        
    hexes = pd.concat(hex_series, ignore_index=True)
    hexes = _enforce_preferred_order(hexes)
    hexes = _drop_nulls(hexes, hex_res)
    hexes.set_index(hex_res, inplace=True)
    hexes.index.name = hex_res
    
    return hexes

In [ ]:
hexes = get_land_use_maps(11)

hexes.head(3)

## Street Network

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings('ignore',
                            message='Iteration over multi-part')

    bh = ox.graph_from_place('Belo Horizonte, MG, Brazil',
                             network_type='drive',
                             simplify=False,)
    
bh = ox.project_graph(bh, to_crs='EPSG:31983')

## OD Matrix and Traffic Zones

In [ ]:
bh_trip_folder = (db_folder
                  / 'beaga'
                  / 'origin_destination_survey'
                  / '2012')

path_to_tzs = bh_trip_folder / 'malha_areas_homogeneas.zip'
path_to_od = bh_trip_folder / 'internal_trips_RMBH.csv'

In [ ]:
def _get_traffic_zones(path, output_epsg):
    if path.suffix == '.zip':
        path = ('zip://'
                + path.as_posix())
    
    # Shapefile's CRS was neither in the data not it was clear:
    # EPSG:4674 (GCS SIRGAS 2000) is my best guess
    geometries = gpd.read_file(str(path),
                               crs=f'EPSG:4674',
                               geometry='geometry',
                              )
    geometries.to_crs(epsg=output_epsg, inplace=True)
    
    geometries.rename(columns={'AH2011': 'tz'},
                      inplace=True,
                     )
    
    # TO DO: unhardcode this and allow for better handling of string labels
    geometries = geometries.astype({'tz': int})
    geometries = geometries.astype({'tz': str})
    
    geometries.set_index('tz', inplace=True)
    
    
    return geometries


In [ ]:
def _get_part_of_interest(od):
    to_keep = {'DS_SH_MOTIVO_ORIGEM': 'purp_origin',
               'Município origem': 'muni_origin',
               'AH origem': 'tz_origin',
               'start_time': 'start_time',
               'travel_time': 'travel_time',
               'end_time': 'end_time',
               'AH destino': 'tz_dest',
               'Município destino': 'muni_dest',
               'DS_SH_MOTIVO_DESTINO': 'purp_dest',
               'Modos agrupados': 'travel_mode',
               'Fator expansão': 'expansion'}

    od = od.reindex(columns=to_keep.keys())
    od.rename(columns=to_keep, inplace=True)
    
    od = od.loc[od.muni_origin == od.muni_dest]
    
    
    return od.drop(columns=['muni_origin', 'muni_dest'])


def _get_od_centers(od, tz, trip_end, output_epsg):
    od = tz.merge(od,
                  left_index=True,
                  right_on=f'tz_{trip_end}')
    
    od = gpd.GeoDataFrame(od,
                          geometry='geometry',
                          crs='EPSG:31983')
    
    od[f'x_{trip_end}'] = od.centroid.x
    od[f'y_{trip_end}'] = od.centroid.y
    
    
    return od.drop(columns='geometry')


def get_od(od_path, tz_path, output_epsg=31983,
           sep=';', encoding='utf-8', decimal=','):
    od = pd.read_csv(od_path,
                     sep=sep,
                     encoding=encoding,
                     decimal=decimal,
                     dtype={'AH origem': str,
                            'AH destino': str},
                     parse_dates={'start_time': ['Hora início'],
                                  'travel_time': ['TEMPO DE DESLOCAMENTO'],
                                  'end_time': ['Hora fim']})
    
    od = _get_part_of_interest(od)
    
    tz = _get_traffic_zones(tz_path, output_epsg)
    od = _get_od_centers(od, tz, 'origin', output_epsg)
    od = _get_od_centers(od, tz, 'dest', output_epsg)
    
    
    return od

In [ ]:
od = get_od(path_to_od, path_to_tzs)

In [ ]:
def project_tz_in_graph(G, od):
    # This should have been done with the traffic
    # zones shapefile. Doing this here incurs
    # into redundant computations
    origin_node, o_dists = ox.nearest_nodes(G,
                                            od.x_origin,
                                            od.y_origin,
                                            return_dist=True)
    od['origin_node'] = origin_node
    od['origin_proj_dist'] = o_dists

    dest_nodes, d_dists = ox.nearest_nodes(G,
                                           od.x_origin,
                                           od.y_origin,
                                           return_dist=True)
    od['dest_node'] = origin_node
    od['dest_proj_dist'] = o_dists
    

In [ ]:
project_tz_in_graph(bh, od)

In [ ]:
od.describe()

In [ ]:
df = od.loc[od.travel_mode == 'coletivo']

q1 = df.travel_time_sec.quantile(.25)
q3 = df.travel_time_sec.quantile(.75)
iq_range = q3 - q1

threshold = q3 + (1.5 * iq_range)

df = df.loc[(df.travel_time_sec > 0)
            & (df.travel_time_sec <= threshold)]

f, ax = plt.subplots(figsize=(20, 7))

sns.histplot(data=df,
            ax=ax,
            x='travel_time_sec')

In [ ]:
od

# Car Acessibility

In order to evaluate car accessibility there needs to be a reasonable estimation of the travel times through street links.
OpenStreetMap data counts with a maximum speed 

In [ ]:
with pd.option_context('mode.chained_assignment', None):
    car_od = od.loc[od.travel_mode == 'individual', :]
    car_od['travel_time_sec'] = (car_od.travel_time.dt.hour * 3600
                                 + car_od.travel_time.dt.minute * 60)
    
    # Estimating travel times for morning peak hours
    car_od = (car_od.set_index('start_time')
                    .between_time('06:00', '08:00')
                    .reset_index())
    
    # Trips internal to traffic zones are useless here,
    # as they allow no routing on the network, since
    # origin point is the same as the destination one
    car_od = car_od.loc[car_od.tz_dest != car_od.tz_origin]
    
    car_od = car_od.reindex(columns=['tz_origin', 'travel_time_sec',
                                     'tz_dest', 'origin_node'
                                     'dest_node'])
    
    car_od = car_od.groupby(['tz_origin', 'tz_dest'], as_index=False).median()
    
car_od.head(3)

In [ ]:
car_od.describe()

A more in depth analysis is required, since 83,640 seconds in impossibly long time, on the one hand, ans a travel time of zero seconds is impossible.

In [ ]:
f, ax = plt.subplots(figsize=(20, 7))

sns.boxplot(ax=ax,
            data=car_od,
            y='travel_time_sec',)
ax.set_yscale('log')

In [ ]:
q1 = car_od.travel_time_sec.quantile(.25)
q3 = car_od.travel_time_sec.quantile(.75)
iq_range = q3 - q1

threshold = q3 + (1.5 * iq_range)

car_od = car_od.loc[(car_od.travel_time_sec > 0)
                    & (car_od.travel_time_sec <= threshold)]

In [ ]:
car_od.describe()[['travel_time_sec']] / 60

That is very much more plausible than the previous travel times

In [ ]:
ox.nearest_nodes(bh, car_od.x_origin, car_od.y_origin, return_dist=True)

In [ ]:
db_folder / 'beaga' / 'GTFS' / 

In [ ]:
ptg

In [ ]:
bh = ox.add_edge_speeds(bh)
bh = ox.add_edge_travel_times(bh)

In [ ]:
def _convert_nodes_df(nodes):
    return nodes.reindex(columns=['x', 'y'])

def _convert_edges_df(edges, use_travel_time=True):
    edges.reset_index(inplace=True)
    
    to_keep = {'u': 'from',
               'v': 'to',
               'travel_time': 'weight',}
    
    if not use_travel_time:
        to_keep.pop('travel_time')
        to_keep['length'] = 'weight'
        
    edges = edges.reindex(columns=list(edges.keys()))
    edges.rename(columns=to_keep, inplace=True)
    
    
    return edges.set_index(['from', 'to'], drop=False)


def osmnx_to_pandana_net(net, use_travel_time=True):
    nodes, edges = ox.graph_to_gdfs(net)
    
    # TO DO: verify if pandana allows variations:
    # i.e. different names, different indexes etc
    edges = _convert_edges_df(edges, use_travel_time)
    nodes = _convert_nodes_df(nodes)
    
    return pdna.Network(nodes['x'],
                        nodes['y'],
                        edges['from'],
                        edges['to'],
                        edges[['weight']])


In [ ]:
bh_net = osmnx_to_pandana_net(bh)

In [ ]:
hex_2011 = hexes.loc[hexes.year == '2011']

In [ ]:
commercial_uses = hex_2011.loc[hex_2011.use.isin(['retail/services', 'mixed'])]

retail_nodes = bh_net.get_node_ids(commercial_uses.x_epsg31983,
                                   commercial_uses.y_epsg31983)

bh_net.set(retail_nodes, name='retail')

In [ ]:
access = bh_net.aggregate(distance=15*60, # seconds
                          type='count',
                          name='retail')

access.name = 'cum_access'

In [ ]:
bh_net.plot(access, 
            fig_kwargs={'figsize': [10, 11]},
            plot_kwargs={'cmap': 'BrBG', 's': 8, 'edgecolor': 'none'})

In [ ]:
nodes, edges = ox.graph_to_gdfs(bh)

In [ ]:
edges.speed_kph.describe()

# Transit Access

In [ ]:
path_to_gtfs = (db_folder
                / 'beaga'
                / 'GTFS'
                / '2017'
                / 'GTFS_BH_2017.01.05.zip')

In [ ]:
state = 31
cities = 3106200

cutoffs = [0,5,9,12,14,17,19,24]

# PROCESSING

In [ ]:
feed = gtfs.load_feed(path_to_gtfs)

In [ ]:
feed.shapes.iloc[0].geometry.coords[:2]

In [ ]:
feed.trips.iloc[0].route_id

In [ ]:
raise Exception(
            "There's only one (apparently) valid transit stop in "\
            f"route {route_id}, which doesn't really make sense."\
                       )

In [ ]:
with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=FutureWarning)
        
        routes = feed.routes
        trips = feed.trips
        stop_times = feed.stop_times
        stops = feed.stops
        shapes = feed.shapes
        
if shapes.empty:
    print('Feed data does not contain route shapes')

trips = (trips.merge(routes, how='left')
              .reindex(columns=['trip_id',
                                'route_id',
                                'service_id',
                                'direction_id',
                                'shape_id'])
        )

stop_times = (stop_times.merge(trips, how='left')
                        .merge(stops, how='left'))

In [ ]:
stop_times

In [ ]:
stop_seq = (stop_times.drop_duplicates(subset=['stop_id','stop_name',
                                                  'stop_sequence', 'shape_id'])
                          .reindex(columns=['route_id','direction_id',
                                            'stop_id','stop_name',
                                            'stop_sequence', 'shape_id'])
               )

In [ ]:
(stop_seq.pivot_table('stop_id',
                                         index=['route_id',
                                                'direction_id',
                                                'shape_id'],
                                         aggfunc='count')
                            .reset_index()
                   )

In [ ]:
feed.trips.loc[feed.trips.shape_id.notnull()]

In [ ]:
bh.nodes

In [ ]:
ox.shortest_path(bh, orig=[8795213893, 7694712699], dest=[8795214055, 8795213762], weight='length')

In [ ]:
bh.nodes[8795213893]

In [ ]:
feed.trips.what

In [ ]:
len(feed.stop_times)

In [ ]:
subset = ['stop_id', 'stop_name', 'stop_sequence', 'shape_id']

col_order = ['route_id', 'direction_id', 'shape_id', 'stop_id',
             'stop_name', 'stop_sequence', 'geometry']

sort_on = ['route_id', 'direction_id', 'shape_id', 'stop_sequence']

stop_sequence = (operations
                 .drop_duplicates(subset=subset)
                 .reindex(columns=col_order)
                 .pipe(gpd.GeoDataFrame,
                       crs='EPSG:4326',
                       geometry='geometry')
                 .sort_values(sort_on)
                 .to_crs(epsg=5641)
                )

In [ ]:
feed.trips

In [ ]:
west, south, east, north  = feed.stops.to_crs(epsg=4326).total_bounds
roads = ox.graph_from_bbox(north,
                           south,
                           east,
                           west,
                           network_type='drive',)

In [ ]:
nodes, edges = ox.graph_to_gdfs(roads)

In [ ]:
list(edges.iloc[0].geometry.coords)

In [ ]:
edges.loc[edges.index==edges.index[0]].reset_index()

In [ ]:
cu[['osmid', 'oneway']] = [np.nan, np.nan]


In [ ]:
roads[27461710][28384475][0]

In [ ]:
mapping = {old: new
           for old, new
           in zip(list(roads.nodes), range(1, len(roads)))}

cu = nx.relabel_nodes(roads, mapping)
           

In [ ]:
nodes, edges = ox.graph_to_gdfs(cu)

In [ ]:
nodes.index

In [ ]:
stops.plot()

In [ ]:
cut_routes_df, anomalies = gtfs.cut_routes(stop_times=stop_times,
                                           route_shapes=shapes,
                                           flag_outliers=True,
                                           threshold=2.5,)

Route shapefiles cannot be properly built because gtfs data is incomplete. I'll have to make do with data from other sources

In [ ]:
map_ = plot_gtfs_data(gtfs_data=[route_summary, stop_times, shapes],
                      variable='trips',
                      window='05:00 - 09:00',
                      direction=None,
                      method='NaturalBreaks',
                      k=5,
                      cmap='magma',
                      linear=False,
                      tiles='cartodbpositron',)

map_

In [ ]:
import os
import pathlib
import re
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import gtfstools as gtfs


%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)
out_folder = out_folder / 'B'

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder)

In [ ]:
path = (db_folder
        / 'beaga'
        / 'GTFS'
        / '2021'
        / 'GTFS_BH_convencional_2021.01.11.zip')

In [ ]:
path = 'GTFS_2017.zip'

In [ ]:
feed = gtfs.load_feed(path)

In [ ]:
summary = gtfs.summarize_trips(feed, summ_by='route_id', cutoffs=[0, 6, 9, 12, 14, 17, 19, 24])

In [ ]:
f, ax = plt.subplots(figsize=(15,7.5))

data = summary.loc[summary.headway_minutes<300]

sns.boxplot(data=data, y='headway_minutes', x='window', ax=ax)

In [ ]:
summary.loc[summary.stop_sequence==1, 'route_id'].value_counts()

In [ ]:
stop_times

In [ ]:
feed.frequencies.sort_values(['trip_id', 'start_time'])

In [ ]:
3540/3600

In [ ]:
feed.stop_times